# Phase 4 — multi-objective BO panel shape optimization (V1-slim Stage 2)

**Dual-mode**: runs on **Colab** (CPU — SU2 is single-threaded) or **locally** (macOS SU2 via Rosetta). Auto-detects.

Thin orchestration only — all logic is tested library code in `fanopt.bo.{codec,objective,inertia,structural,backbone,orchestration}` + `scripts/run_phase4_bo.py` (CLAUDE.md §6). Each design: decode → Path A+ 2D slice → unsteady SU2 → `J_fan`, plus CadQuery `I_wrist` and plate-bending panel stiffness. The **qLogNEHVI + TuRBO** campaign returns the 3-objective Pareto front (airflow ↑, inertia ↓, deflection ↓).


## 1. Repo + Python/native deps


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    REPO_DIR = Path("/content/fan-optimization")
    BRANCH = "main"
    REPO_URL = "https://github.com/clingergab/fan-optimization.git"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    # gmsh dlopens libGLU + X libs at import; Colab CPU runtimes lack them.
    subprocess.run(
        "apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
        "libxft2 libxinerama1 unzip".split(),
        check=True,
    )
    # Phase 4 needs the BO stack (botorch) on top of the geometry/CFD deps.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "gmsh", "cadquery", "scipy", "jinja2", "pyyaml", "botorch", "tqdm", "ipywidgets"],
        check=True,
    )
else:
    REPO_DIR = Path.cwd()
    while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "pyproject.toml").exists():
        REPO_DIR = REPO_DIR.parent

for p in (REPO_DIR / "src", REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

assert (REPO_DIR / "scripts" / "run_phase4_bo.py").exists(), \
    "phase-4 code missing — is the branch pushed?"

# Preflight: Phase 4 needs cadquery (I_wrist) + gmsh (slice mesh) + botorch (BO).
# Fail here with a clear message rather than a cryptic ModuleNotFoundError deep in
# an import chain when the notebook is running on the wrong kernel/interpreter.
_missing = [m for m in ("cadquery", "gmsh", "botorch", "scipy", "yaml")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise ModuleNotFoundError(
        f"Missing {_missing} in this kernel:\n  {sys.executable}\n"
        "Phase 4 needs cadquery + gmsh + botorch + scipy + pyyaml.\n"
        "  • Local (VSCode): pick the kernel whose interpreter is the one with these "
        "installed (e.g. 'fanopt-local' → /usr/local/bin/python3), or "
        "`<that-python> -m pip install -e '.[bo]' cadquery gmsh`.\n"
        "  • Colab: re-run this cell — the IN_COLAB branch pip-installs them."
    )
print("repo:", REPO_DIR, "| colab:", IN_COLAB, "| deps OK:", sys.executable)


## 2. SU2 — install (Colab) or locate (local)


In [ ]:
import importlib.util
import os
import subprocess
import urllib.request
from pathlib import Path

if importlib.util.find_spec("google.colab") is not None:
    SU2_VERSION = "8.0.1"
    SU2_DIR = Path("/content/su2")
    if not any(SU2_DIR.rglob("SU2_CFD")):
        url = (
            f"https://github.com/su2code/SU2/releases/download/"
            f"v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip"
        )
        print("[su2] downloading", url)
        urllib.request.urlretrieve(url, "/tmp/su2.zip")
        SU2_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/su2.zip", "-d", str(SU2_DIR)], check=True)
    cands = [p for p in SU2_DIR.rglob("SU2_CFD") if p.is_file() and not p.is_symlink()]
    assert cands, "SU2_CFD not found after extract — see colab_spike_0_6c.ipynb"
    bindir = cands[0].parent
    for p in bindir.iterdir():
        if p.is_file():
            os.chmod(p, 0o755)
else:
    bindir = Path.home() / "su2-local" / "extracted" / "bin"
    assert (bindir / "SU2_CFD").exists(), f"SU2_CFD not at {bindir}"

os.environ["SU2_RUN"] = str(bindir)
os.environ["PATH"] = str(bindir) + os.pathsep + os.environ.get("PATH", "")
print("SU2_RUN:", os.environ["SU2_RUN"])


## 3. Campaign settings + persistence

Configured for a **production 8-core run**: `USE_PRODUCTION=True` (5-cycle / dt=T/200), `N_WORKERS=8` + `BATCH_SIZE=8` so both the DoE and the BO loop run 8-wide (~208 evals, roughly 4–5 h). For a quick smoke instead, set `USE_PRODUCTION=False`, `N_INIT=4`, `N_ITERATIONS=1`.

On **Colab** this mounts Google Drive and puts the campaign there, so a long run **survives a session disconnect** — just re-run the Run cell to resume from the last checkpoint.


In [ ]:
from pathlib import Path

# --- Campaign size (production run) ----------------------------------------
# A production eval is ~8-12 min (5 cycles, dt=T/200). With 8-way parallelism
# below, wall time ≈ ceil(evals / N_WORKERS) * per_eval. The settings here are
# ~208 evals → ~26 parallel rounds → roughly 4-5 h. It checkpoints every batch,
# so a Colab disconnect is fine — just re-run this + the Run cell to resume.
N_INIT = 16              # DoE seed (2 full 8-wide batches) for the 35-var space
N_ITERATIONS = 24        # BO refinement iterations (× BATCH_SIZE evals each)
SEED = 0
STALL_PATIENCE = 8
USE_PRODUCTION = True     # locked 5-cycle / dt=T/200 unsteady resolution

# --- Throughput (Colab 8-core CPU runtime) ---------------------------------
# CFD evals run in separate PROCESSES (gmsh can't be threaded). N_WORKERS runs
# one eval per core; BATCH_SIZE = N_WORKERS makes qNEHVI propose a full batch so
# the *dominant BO loop* parallelizes too, not just the DoE.
N_WORKERS = 8            # one CFD process per core (drop this if you hit OOM)
BATCH_SIZE = N_WORKERS   # q-batch BO → DoE and loop both run 8-wide

# --- Persistence -----------------------------------------------------------
# On Colab, put the campaign dir on Google Drive so checkpoint.npz + the ledger
# survive a session disconnect: re-running the RUN cell resumes from the last
# completed evaluation (run_campaign resume=True). Locally it stays under data/.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/fanopt/phase4_bo")
else:
    WORK = REPO_DIR / "data" / "phase4_bo_nb"
WORK.mkdir(parents=True, exist_ok=True)
print("campaign dir:", WORK, "| production:", USE_PRODUCTION,
      "| plan:", N_INIT, "seeds +", N_ITERATIONS, "iters")


## 4. Run the BO campaign

Safe to **re-run after a disconnect** — `resume=True` continues from the Drive checkpoint instead of restarting.


In [ ]:
import run_phase4_bo as script
from fanopt.bo.objective import PRODUCTION_EVAL_CFG
from fanopt.bo.orchestration import CampaignConfig

cfg = CampaignConfig(
    n_init=N_INIT, n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
    seed=SEED, stall_patience=STALL_PATIENCE, n_workers=N_WORKERS,
    num_restarts=8, raw_samples=128, mc_samples=128,
)
# resume=True (default): safe to re-run after a Colab disconnect — it continues
# from the Drive checkpoint instead of restarting.
summary = script.run(
    out_dir=WORK, cfg=cfg, su2_bin=None,
    eval_cfg=PRODUCTION_EVAL_CFG if USE_PRODUCTION else None,
)
print("evals =", summary["n_evaluations"],
      "| pareto =", summary["n_pareto"],
      "| fallback =", summary["used_fallback"])
summary["pareto"][:2]


## 5. Pareto front


In [ ]:
import json
import matplotlib.pyplot as plt

data = json.loads((WORK / "pareto.json").read_text())
pf = data["pareto"]
j = [d["j_fan"] for d in pf]
iw = [d["i_wrist_kgm2"] for d in pf]
st = [d["structural_m"] * 1000.0 for d in pf]  # mm

fig, ax = plt.subplots(figsize=(5.6, 4.2))
sc = ax.scatter(j, iw, c=st, s=80, cmap="viridis")
for d in pf:
    ax.annotate(f"b{d['blade_count']}", (d["j_fan"], d["i_wrist_kgm2"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("J_fan  (maximize →)")
ax.set_ylabel("I_wrist  kg·m²  (← minimize)")
fig.colorbar(sc, label="panel deflection (mm, minimize)")
ax.set_title(f"Phase 4 Pareto — {data['n_pareto']} designs / {data['n_evaluations']} evals")
fig.tight_layout()
plt.show()
